# Märklin DC Motor PWM & BEMF Simulation Engine

This notebook simulates the electromechanical dynamic behavior of DC motors (e.g., Märklin model train motors) driven by PWM, incorporating Back-EMF (bEMF) sensing gaps and commutator ripple detection.

In [1]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## 1. Core Physics Simulation Model

The function  models the coupled electrical and mechanical differential equations for a DC motor under PWM excitation with high-impedance bEMF measurement gaps.

In [2]:
import numpy as np

def simulate_dc_motor(
    n_coils=3,
    v_in=15.0,
    pwm_freq=1000.0,
    pwm_duty=50.0,
    pwm_duration_ms=200.0,
    bemf_window_ms=2.0,
    bemf_sample_rate_hz=50000.0,
    gap_interval_ms=50.0,
    R=8.0,
    L=0.005,
    Ke=0.015,
    Kt=0.015,
    J=1e-5,
    B=1e-5,
    T_coulomb=0.001,
    ripple_amplitude=0.08,
    comm_spike_amplitude=0.20,
    comm_spike_width=0.10,
    brush_noise_amplitude=0.02,
    dt=2e-6
):
    """
    Simulates DC motor electromechanical dynamics under PWM drive with periodic bEMF gaps
    and realistic commutator current spikes/brush arcing noise.
    """
    total_time = pwm_duration_ms / 1000.0
    num_steps = int(total_time / dt)
    
    t_arr = np.linspace(0, total_time, num_steps)
    v_m_arr = np.zeros(num_steps)
    i_arr = np.zeros(num_steps)
    w_arr = np.zeros(num_steps)
    theta_arr = np.zeros(num_steps)
    is_gap_arr = np.zeros(num_steps, dtype=bool)
    
    i = 0.0
    w = 0.0
    theta = 0.0
    
    pwm_period = 1.0 / pwm_freq
    gap_interval = gap_interval_ms / 1000.0
    bemf_window = bemf_window_ms / 1000.0
    comm_period = np.pi / n_coils
    
    for idx, t in enumerate(t_arr):
        time_in_gap_cycle = t % gap_interval
        is_gap = (time_in_gap_cycle < bemf_window) and (t > 0.005) # start gaps after initial startup
        is_gap_arr[idx] = is_gap
        
        if is_gap:
            v_applied = 0.0
            hbridge_state = 'FLOAT'
        else:
            time_in_pwm_cycle = t % pwm_period
            pwm_on_time = (pwm_duty / 100.0) * pwm_period
            if time_in_pwm_cycle < pwm_on_time:
                v_applied = v_in
                hbridge_state = 'DRIVE'
            else:
                v_applied = 0.0
                hbridge_state = 'OFF'
        
        # Commutator angular position calculation
        phi = (theta % comm_period) / comm_period
        dist_to_trans = np.minimum(phi, 1.0 - phi)
        comm_spike = comm_spike_amplitude * np.exp(- (dist_to_trans / (comm_spike_width / 2.0))**2) if comm_spike_width > 0 else 0.0
        brush_noise = brush_noise_amplitude * (0.6 * np.sin(127.0 * theta) + 0.4 * np.cos(311.0 * t))
        
        ripple_freq_factor = 2.0 * n_coils
        ripple_mod = 1.0 + ripple_amplitude * np.cos(ripple_freq_factor * theta)
        
        # Commutation bridging reduces effective resistance & bEMF temporarily during coil shorting
        spike_factor = (comm_spike / comm_spike_amplitude) if comm_spike_amplitude > 0 else 0.0
        e_back = Ke * w * (ripple_mod - 0.25 * spike_factor)
        R_eff = R * (1.0 + 0.05 * np.sin(ripple_freq_factor * theta) - 0.15 * spike_factor)
        
        if hbridge_state == 'DRIVE':
            v_motor = v_applied
            di_dt = (v_motor - e_back - i * R_eff) / L
        elif hbridge_state == 'OFF':
            if i > 0.0005:
                v_motor = -0.7
                di_dt = (v_motor - e_back - i * R_eff) / L
            else:
                i = 0.0
                v_motor = e_back
                di_dt = 0.0
        elif hbridge_state == 'FLOAT':
            if i > 0.0005:
                v_motor = -0.7
                di_dt = (v_motor - e_back - i * R_eff) / L
            else:
                i = 0.0
                v_motor = e_back
                di_dt = 0.0
                
        i += di_dt * dt
        if i < 0:
            i = 0.0
            
        T_elec = Kt * i * ripple_mod
        T_fric = B * w + (T_coulomb if w > 0.001 else 0.0)
        dw_dt = (T_elec - T_fric) / J
        
        w += dw_dt * dt
        if w < 0:
            w = 0.0
            
        theta += w * dt
        
        # Terminal current measured at shunt includes brush commutation surge & contact noise
        i_measured = i * (1.0 + comm_spike + brush_noise) if i > 0.0001 else 0.0
        
        v_m_arr[idx] = v_motor
        i_arr[idx] = i_measured
        w_arr[idx] = w
        theta_arr[idx] = theta
        
    # Sample ADC readings during gaps
    bemf_adc_dt = 1.0 / bemf_sample_rate_hz
    gap_indices = np.where(is_gap_arr)[0]
    
    adc_timestamps = []
    adc_bemf_readings = []
    
    if len(gap_indices) > 0:
        gap_times = t_arr[gap_indices]
        current_sample_time = gap_times[0]
        end_time = gap_times[-1]
        
        while current_sample_time <= end_time:
            # find closest index
            idx = int(current_sample_time / dt)
            if idx < num_steps and is_gap_arr[idx]:
                adc_timestamps.append(t_arr[idx])
                adc_bemf_readings.append(v_m_arr[idx])
            current_sample_time += bemf_adc_dt

    return {
        't': t_arr,
        'v_m': v_m_arr,
        'i': i_arr,
        'w': w_arr,
        'theta': theta_arr,
        'is_gap': is_gap_arr,
        'adc_t': np.array(adc_timestamps),
        'adc_bemf': np.array(adc_bemf_readings)
    }

## 2. Motor Characteristics Presets

Pre-configured parameter sets tailored for various locomotive motor types (Alte Märklinmotoren, Glockenankermotor, Moderne Motoren, and Grosse Gartenbahnmotoren).

In [3]:
MOTOR_PRESETS = {
    'Alte Märklinmotoren (3-Pol LFCM/SFCM/DCM)': {
        'n_coils': 3,
        'v_in': 15.0,
        'pwm_freq': 1000.0,
        'pwm_duty': 50.0,
        'pwm_duration_ms': 200.0,
        'bemf_window_ms': 2.0,
        'bemf_sample_rate_hz': 50000.0,
        'R': 7.5,
        'L': 0.012,
        'Ke': 0.018,
        'Kt': 0.018,
        'J': 3e-5,
        'B': 2e-5,
        'T_coulomb': 0.002,
        'ripple_amplitude': 0.12,
        'comm_spike_amplitude': 0.35,
        'comm_spike_width': 0.12,
        'brush_noise_amplitude': 0.04,
        'description': 'Classic heavy iron armature Märklin motor with high inductance, pronounced commutator ripple, and high brush commutation spikes.'
    },
    'Glockenankermotor (Faulhaber / Maxon)': {
        'n_coils': 5,
        'v_in': 12.0,
        'pwm_freq': 16000.0,
        'pwm_duty': 50.0,
        'pwm_duration_ms': 200.0,
        'bemf_window_ms': 1.0,
        'bemf_sample_rate_hz': 50000.0,
        'R': 15.0,
        'L': 0.0003,
        'Ke': 0.012,
        'Kt': 0.012,
        'J': 2e-7,
        'B': 3e-6,
        'T_coulomb': 0.0002,
        'ripple_amplitude': 0.03,
        'comm_spike_amplitude': 0.08,
        'comm_spike_width': 0.05,
        'brush_noise_amplitude': 0.005,
        'description': 'Coreless motor with ultra-low inductance, smooth precious metal commutation, and minimal arcing.'
    },
    'Moderne Motoren (5-Pol HO DC Can)': {
        'n_coils': 5,
        'v_in': 15.0,
        'pwm_freq': 4000.0,
        'pwm_duty': 50.0,
        'pwm_duration_ms': 200.0,
        'bemf_window_ms': 2.0,
        'bemf_sample_rate_hz': 50000.0,
        'R': 12.0,
        'L': 0.003,
        'Ke': 0.015,
        'Kt': 0.015,
        'J': 3e-6,
        'B': 1e-5,
        'T_coulomb': 0.0008,
        'ripple_amplitude': 0.06,
        'comm_spike_amplitude': 0.20,
        'comm_spike_width': 0.08,
        'brush_noise_amplitude': 0.02,
        'description': 'Standard modern 5-pole enclosed DC can motor used in current locomotives.'
    },
    'Grosse Gartenbahnmotoren (LGB / Spur I)': {
        'n_coils': 5,
        'v_in': 18.0,
        'pwm_freq': 1000.0,
        'pwm_duty': 50.0,
        'pwm_duration_ms': 200.0,
        'bemf_window_ms': 2.5,
        'bemf_sample_rate_hz': 50000.0,
        'R': 4.0,
        'L': 0.020,
        'Ke': 0.035,
        'Kt': 0.035,
        'J': 1.5e-4,
        'B': 5e-5,
        'T_coulomb': 0.005,
        'ripple_amplitude': 0.08,
        'comm_spike_amplitude': 0.25,
        'comm_spike_width': 0.10,
        'brush_noise_amplitude': 0.03,
        'description': 'Large, heavy motor for garden railways (G scale / Gauge 1) with high torque, heavy current surges, and strong commutation spikes.'
    }
}

## 3. Interactive UI & Simulation Plotting Dashboard

Use the dropdown to select pre-configured motor profiles or adjust parameters dynamically using the sliders below.

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

slider_style = {'description_width': 'initial'}

# Default controls initialization
preset_dropdown = widgets.Dropdown(
    options=list(MOTOR_PRESETS.keys()),
    value='Alte Märklinmotoren (3-Pol LFCM/SFCM/DCM)',
    description='Preset:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

w_n_coils = widgets.IntSlider(value=3, min=3, max=12, step=1, style=slider_style, description='Coils (n):')
w_v_in = widgets.FloatSlider(value=15.0, min=3.0, max=24.0, step=0.5, style=slider_style, description='V_in (V):')
w_pwm_freq = widgets.FloatLogSlider(value=1000.0, base=10, min=2, max=4.5, step=0.05, style=slider_style, description='PWM Freq (Hz):')
w_pwm_duty = widgets.FloatSlider(value=50.0, min=0.0, max=100.0, step=1.0, style=slider_style, description='Duty (%):')
w_pwm_duration = widgets.FloatSlider(value=200.0, min=20.0, max=1000.0, step=10.0, style=slider_style, description='Duration (ms):')
w_bemf_window = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.1, style=slider_style, description='bEMF Window (ms):')
w_bemf_rate = widgets.FloatSlider(value=50000.0, min=10000.0, max=200000.0, step=5000.0, style=slider_style, description='Sampling (Hz):')

w_R = widgets.FloatSlider(value=7.5, min=1.0, max=30.0, step=0.5, style=slider_style, description='R (Ohm):')
w_L = widgets.FloatSlider(value=12.0, min=0.1, max=50.0, step=0.1, style=slider_style, description='L (mH):')
w_Ke = widgets.FloatSlider(value=0.018, min=0.001, max=0.08, step=0.001, style=slider_style, description='Ke (V/(rad/s)):')
w_J = widgets.FloatSlider(value=30.0, min=0.1, max=200.0, step=0.1, style=slider_style, description='J (1e-6 kg m^2):')
w_ripple = widgets.FloatSlider(value=0.12, min=0.0, max=0.3, step=0.01, style=slider_style, description='Ripple Factor:')
w_comm_spike_amp = widgets.FloatSlider(value=0.35, min=0.0, max=0.8, step=0.02, style=slider_style, description='Comm Spike Amp:')
w_comm_spike_width = widgets.FloatSlider(value=0.12, min=0.01, max=0.3, step=0.01, style=slider_style, description='Comm Spike Width:')
w_brush_noise = widgets.FloatSlider(value=0.04, min=0.0, max=0.15, step=0.005, style=slider_style, description='Brush Noise:')

description_out = widgets.HTML(value=f"<b>Description:</b> {MOTOR_PRESETS[preset_dropdown.value]['description']}")

def on_preset_change(change):
    p = MOTOR_PRESETS[change['new']]
    w_n_coils.value = p['n_coils']
    w_v_in.value = p['v_in']
    w_pwm_freq.value = p['pwm_freq']
    w_pwm_duty.value = p['pwm_duty']
    w_pwm_duration.value = p['pwm_duration_ms']
    w_bemf_window.value = p['bemf_window_ms']
    w_bemf_rate.value = p['bemf_sample_rate_hz']
    w_R.value = p['R']
    w_L.value = p['L'] * 1000.0
    w_Ke.value = p['Ke']
    w_J.value = p['J'] * 1e6
    w_ripple.value = p['ripple_amplitude']
    w_comm_spike_amp.value = p.get('comm_spike_amplitude', 0.20)
    w_comm_spike_width.value = p.get('comm_spike_width', 0.10)
    w_brush_noise.value = p.get('brush_noise_amplitude', 0.02)
    description_out.value = f"<b>Description:</b> {p['description']}"

preset_dropdown.observe(on_preset_change, names='value')

out = widgets.Output()

def run_and_plot_simulation(b=None):
    with out:
        clear_output(wait=True)
        res = simulate_dc_motor(
            n_coils=w_n_coils.value,
            v_in=w_v_in.value,
            pwm_freq=w_pwm_freq.value,
            pwm_duty=w_pwm_duty.value,
            pwm_duration_ms=w_pwm_duration.value,
            bemf_window_ms=w_bemf_window.value,
            bemf_sample_rate_hz=w_bemf_rate.value,
            R=w_R.value,
            L=w_L.value / 1000.0,
            Ke=w_Ke.value,
            Kt=w_Ke.value,
            J=w_J.value / 1e6,
            ripple_amplitude=w_ripple.value,
            comm_spike_amplitude=w_comm_spike_amp.value,
            comm_spike_width=w_comm_spike_width.value,
            brush_noise_amplitude=w_brush_noise.value
        )
        
        t_ms = res['t'] * 1000.0
        v_m = res['v_m']
        i_m = res['i']
        rpm = res['w'] * 60.0 / (2.0 * np.pi)
        
        fig, axs = plt.subplots(3, 1, figsize=(11, 10), sharex=False)
        fig.suptitle('Märklin DC Motor PWM Drive & bEMF Readout Simulation', fontsize=14, fontweight='bold')
        
        # Plot 1: Overall Overview (Voltage & Current & Speed)
        ax1 = axs[0]
        ax1.plot(t_ms, v_m, label='Motor Terminal Voltage (V)', color='tab:blue', alpha=0.8)
        ax1.set_ylabel('Voltage (V)', color='tab:blue')
        ax1.set_title('Global Overview: Voltage, Current, and Speed (RPM)')
        ax1.grid(True, linestyle='--', alpha=0.6)
        
        ax1_i = ax1.twinx()
        ax1_i.plot(t_ms, i_m, label='Current (A)', color='tab:orange', alpha=0.7)
        ax1_i.set_ylabel('Current (A)', color='tab:orange')
        
        # Plot 2: Speed Trajectory
        ax2 = axs[1]
        ax2.plot(t_ms, rpm, label='Rotor Speed (RPM)', color='tab:green', linewidth=2)
        ax2.set_xlabel('Time (ms)')
        ax2.set_ylabel('Speed (RPM)')
        ax2.set_title('Rotor Speed Response')
        ax2.grid(True, linestyle='--', alpha=0.6)
        ax2.legend(loc='upper left')
        
        # Plot 3: Zoomed bEMF Gap & ADC Sampling
        ax3 = axs[2]
        gap_indices = np.where(res['is_gap'])[0]
        if len(gap_indices) > 0:
            # Pick a gap in steady state (middle gap)
            gap_times = res['t'][gap_indices]
            mid_gap_t = gap_times[len(gap_times) // 2]
            zoom_mask = (res['t'] >= mid_gap_t - 0.001) & (res['t'] <= mid_gap_t + (w_bemf_window.value/1000.0) + 0.002)
            
            t_zoom_ms = res['t'][zoom_mask] * 1000.0
            v_zoom = res['v_m'][zoom_mask]
            i_zoom = res['i'][zoom_mask]
            
            ax3.plot(t_zoom_ms, v_zoom, label='Terminal Voltage (bEMF Gap)', color='tab:purple', linewidth=2)
            ax3.plot(t_zoom_ms, i_zoom * 10, label='Current (x10 A)', color='tab:orange', linestyle='--')
            
            # Plot ADC sampled bEMF points
            if len(res['adc_t']) > 0:
                adc_mask = (res['adc_t'] >= min(res['t'][zoom_mask])) & (res['adc_t'] <= max(res['t'][zoom_mask]))
                ax3.scatter(res['adc_t'][adc_mask] * 1000.0, res['adc_bemf'][adc_mask], color='red', zorder=5, label='ADC Sample (50kHz)')
                
            ax3.set_title(f'Zoomed bEMF Readout Window (Gap Size: {w_bemf_window.value:.1f} ms, Sampling Rate: {w_bemf_rate.value/1000:.0f} kHz)')
            ax3.set_xlabel('Time (ms)')
            ax3.set_ylabel('Voltage (V)')
            ax3.grid(True, linestyle='--', alpha=0.6)
            ax3.legend(loc='upper right')
        else:
            ax3.text(0.5, 0.5, 'No bEMF Gap Detected in Duration', ha='center', va='center')
            
        plt.tight_layout()
        plt.show()

btn_run = widgets.Button(description='Run Simulation', button_style='primary', icon='play')
btn_run.on_click(run_and_plot_simulation)

ui_box = widgets.VBox([
    preset_dropdown,
    description_out,
    widgets.HBox([
        widgets.VBox([widgets.HTML('<b>Operating Parameters</b>'), w_n_coils, w_v_in, w_pwm_freq, w_pwm_duty, w_pwm_duration, w_bemf_window, w_bemf_rate]),
        widgets.VBox([widgets.HTML('<b>Motor Model Parameters</b>'), w_R, w_L, w_Ke, w_J, w_ripple, w_comm_spike_amp, w_comm_spike_width, w_brush_noise])
    ]),
    btn_run,
    out
])

display(ui_box)
# Initial execution
run_and_plot_simulation()
